Read sales_orders, convert types, filter invalid records, save as bronze_orders 

In [0]:
from pyspark.sql.functions import col, from_unixtime, to_timestamp 

 

# Configuration 

source_table = "dbacademy.healthcare.sales_orders" 

target_catalog = "dbacademy" 

target_schema = "healthcare" 

target_table = f"{target_catalog}.{target_schema}.bronze_orders" 

 

In [0]:
# Read source data 

df = spark.read.table(source_table) 

 

# Convert data types: 

# customer_id -> LONG (already LONG, cast to ensure) 

# order_datetime -> TIMESTAMP (from epoch seconds) 

df_transformed = df.select( 

    col("customer_id").cast("long"), 

    col("customer_name"), 

    col("order_number"), 

    to_timestamp(from_unixtime(col("order_datetime"))).alias("order_datetime"), 

    col("number_of_line_items"), 

    col("ordered_products"), 

    col("promo_info"), 

    col("clicked_items") 

) 


In [0]:

 

# Filter out invalid records where customer_id is null 

df_cleaned = df_transformed.filter(col("customer_id").isNotNull()) 

In [0]:

# Save as bronze_orders table 

df_cleaned.write.mode("overwrite").saveAsTable(target_table) 

In [0]:

print(f"Successfully saved bronze_orders to {target_table}") 
print(f"Record count: {spark.read.table(target_table).count()}") 